In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset
import numpy as np
import pandas as pd


In [2]:
# Reading data
read_data_005=pd.read_csv("D:/Man/Earth and Environment/Manchester_data003-008/005_manchester_processed.csv")
read_data_005=read_data_005.sort_values(by='time').reset_index(drop=True)

learning_data_005=read_data_005[(read_data_005['year']>=2006)&(read_data_005['year']<=2050)].copy()
learning_data_005=learning_data_005.drop(columns=['lat','lon','date'],errors='ignore')

learning_data_005

,time,TREFMXAV_U,FLNS,FSNS,PRECT,PRSN,QBOT,TREFHT,UBOT,VBOT,year,month,day,dayofyear
0,2006-01-02,11.15260,8.737288,7.855509,1.544200e-07,4.051599e-16,0.005373,6.07195,2.303055,4.502803,2006,1,2,2
1,2006-01-03,13.01327,6.686464,7.501073,7.784098e-08,0.000000e+00,0.007595,11.04843,4.657475,3.158464,2006,1,3,3
2,2006-01-04,13.16030,27.445148,12.188718,4.851411e-08,7.075068e-18,0.005667,9.27023,5.083677,5.835154,2006,1,4,4
3,2006-01-05,14.85882,10.443632,4.354691,1.091676e-07,1.429017e-14,0.007362,11.82208,3.029053,8.031938,2006,1,5,5
4,2006-01-06,10.78576,70.927300,31.532597,2.531008e-09,6.418103e-18,0.004160,7.14828,3.783906,6.986506,2006,1,6,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15811,2050-12-27,10.58290,19.547508,15.106448,2.213155e-08,6.438513e-15,0.005779,8.01800,0.623059,4.167736,2050,12,361,361
15812,2050-12-28,14.16284,18.568079,13.015922,3.699803e-08,9.786361e-23,0.006798,9.50298,-1.815245,4.800050,2050,12,362,362
15813,2050-12-29,16.11770,15.843533,8.881753,2.557600e-08,1.473843e-22,0.008161,13.01970,-2.800902,7.744351,2050,12,363,363
15814,2050-12-30,13.72466,21.278680,9.996034,4.276372e-08,1.328554e-12,0.006569,10.77773,-1.253943,6.456124,2050,12,364,364


In [3]:
# Data overview
print('-----------DATA DESCRIPTION-----------')
print(learning_data_005.describe())
print('-----------MISSING VALUES-----------')
print(learning_data_005.isnull().sum())

-----------DATA DESCRIPTION-----------
         TREFMXAV_U          FLNS          FSNS         PRECT          PRSN  \
count  15816.000000  15816.000000  15816.000000  1.581600e+04  1.581600e+04   
mean      15.026336     42.627381     93.192801  3.398662e-08  3.744736e-10   
std        4.966141     21.565222     73.114148  4.989475e-08  5.093579e-09   
min        0.918600      0.829912      2.245905  0.000000e+00  0.000000e+00   
25%       11.271578     25.714433     29.885597  2.198835e-09  0.000000e+00   
50%       14.543420     40.400549     74.286275  1.350487e-08  4.558962e-20   
75%       18.900795     56.682670    143.040408  4.702735e-08  5.775253e-16   
max       32.473840    114.220130    308.308960  5.838473e-07  2.661317e-07   

               QBOT        TREFHT          UBOT          VBOT          year  \
count  15816.000000  15816.000000  15816.000000  15816.000000  15816.000000   
mean       0.006291     11.128763      1.119279      1.538834   2027.951189   
std        0

In [4]:
from sklearn.preprocessing import StandardScaler
features=['FLNS','FSNS','PRECT','PRSN','QBOT','TREFHT','UBOT','VBOT','year','month','dayofyear']
X_005=learning_data_005[features].values
y_005=learning_data_005['TREFMXAV_U'].values

scaler_X=StandardScaler()
X_scaled_005=scaler_X.fit_transform(X_005)


In [5]:
# Time rolling windows
train_years = 5
val_years = 3
stride_years = 5

days_per_year = 365

train_size = train_years * days_per_year
val_size = val_years * days_per_year
stride_size = stride_years * days_per_year


def sliding_split(X, y, df, train_size, val_size, stride_size):
    splits = []

    start = 0
    total = len(X)

    while start + train_size + val_size <= total:

        train_end = start + train_size
        val_end = train_end + val_size

        X_train = X[start:train_end]
        y_train = y[start:train_end]

        X_val = X[train_end:val_end]
        y_val = y[train_end:val_end]

        # record time
        train_start_year = df.iloc[start]['year']
        train_end_year = df.iloc[train_end - 1]['year']

        val_start_year = df.iloc[train_end]['year']
        val_end_year = df.iloc[val_end - 1]['year']

        splits.append((
            X_train, y_train, X_val, y_val,
            train_start_year, train_end_year,
            val_start_year, val_end_year
        ))

        start += stride_size

    return splits


splits_005 = sliding_split(X_scaled_005, y_005, learning_data_005, train_size, val_size, stride_size)

print("Windows：", len(splits_005))

Windows： 8


In [6]:
# Windowing
def create_sequences(X,y,window_size=7):
    X_seq,y_seq=[],[]
    X_values=np.array(X).astype(np.float32)
    y_values=np.array(y).astype(np.float32)
    
    for i in range(len(X)-window_size):
        X_seq.append(X[i:i+window_size])
        y_seq.append(y[i+window_size])

    X_tensor=torch.FloatTensor(np.array(X_seq)).transpose(1,2)
    y_tensor=torch.FloatTensor(np.array(y_seq)).view(-1,1)
    return X_tensor,y_tensor

In [7]:
class EarlyStopping:
    def __init__(self, patience=10, verbose=False, delta=0):
        self.patience = patience   
        self.verbose = verbose
        self.counter = 0  
        self.best_score = None  
        self.early_stop = False 
        self.val_loss_min = np.inf
        self.delta = delta  

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):

        self.best_state_dict = model.state_dict()
        self.val_loss_min = val_loss

# 1D-CNN
out channel=32

window size=30

learning rate=0.001

epoch=200

patience=15

In [8]:
# Define 1D-CNN
class TemperatureCNN(nn.Module):
    def __init__(self, input_channels, window_size):
        super(TemperatureCNN,self).__init__()

        self.conv_layer=nn.Sequential(
            nn.Conv1d(in_channels=input_channels,out_channels=32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2) if window_size>2 else nn.Identity(),
)
        self.flatten_dim=32*(window_size//2 if window_size>2 else window_size)
        self.fc_layer=nn.Sequential(
            nn.Linear(self.flatten_dim,32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32,1)
            )
        
    def forward(self,x):
        x=self.conv_layer(x)
        x=x.view(x.size(0),-1)
        x=self.fc_layer(x)
        return x


In [20]:
# Training
import torch.nn.functional as F
window_size=30
input_channels=X_scaled_005.shape[1]
trained_models=[]

all_mae = []
all_rmse = []

for i,(X_tr,y_tr,X_va,y_va,t_s,t_e,v_s,v_e) in enumerate(splits_005):

    X_train_t,y_train_t=create_sequences(X_tr, y_tr,window_size)
    X_val_t,y_val_t=create_sequences(X_va, y_va,window_size)

    train_loader=DataLoader(TensorDataset(X_train_t,y_train_t),batch_size=32,shuffle=True)
    
    model=TemperatureCNN(input_channels,window_size)
    criterion=nn.L1Loss()
    optimizer=optim.Adam(model.parameters(),lr=0.001)

    early_stopping = EarlyStopping(patience=15, verbose=False)

    for epoch in range(200):
        model.train()
        for batch_X,batch_y in train_loader:
            optimizer.zero_grad()
            outputs=model(batch_X)
            loss=criterion(outputs,batch_y)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val_t)
            current_val_mae = criterion(val_outputs, y_val_t).item()
        
        early_stopping(current_val_mae, model)
        
        if early_stopping.early_stop:
            print(f"Split {i+1}: Early stopping at epoch {epoch}")
            break

    model.load_state_dict(early_stopping.best_state_dict)
    model.eval()
    trained_models.append(model)
    
    with torch.no_grad():
        val_preds=model(X_val_t)
        mae=criterion(val_preds,y_val_t).item()
        mse=F.mse_loss(val_preds,y_val_t).item()
        rmse=np.sqrt(mse)

        all_mae.append(mae)
        all_rmse.append(rmse)

    print(f"===== Split {i+1} =====")
    print(f"Train: {int(t_s)} → {int(t_e)}")
    print(f"Val  : {int(v_s)} → {int(v_e)}")
    print(f"Train size: {len(X_tr)} | Val size: {len(X_va)}")
    print(f"MAE: {mae:.3f}, RMSE: {rmse:.3f}")
    print()




Split 1: Early stopping at epoch 47
===== Split 1 =====
Train: 2006 → 2011
Val  : 2011 → 2014
Train size: 1825 | Val size: 1095
MAE: 1.844, RMSE: 2.364

Split 2: Early stopping at epoch 25
===== Split 2 =====
Train: 2011 → 2016
Val  : 2016 → 2019
Train size: 1825 | Val size: 1095
MAE: 2.532, RMSE: 3.087

Split 3: Early stopping at epoch 61
===== Split 3 =====
Train: 2016 → 2021
Val  : 2021 → 2024
Train size: 1825 | Val size: 1095
MAE: 1.884, RMSE: 2.415

Split 4: Early stopping at epoch 83
===== Split 4 =====
Train: 2021 → 2026
Val  : 2026 → 2029
Train size: 1825 | Val size: 1095
MAE: 1.807, RMSE: 2.264

Split 5: Early stopping at epoch 47
===== Split 5 =====
Train: 2026 → 2031
Val  : 2031 → 2035
Train size: 1825 | Val size: 1095
MAE: 1.739, RMSE: 2.206

Split 6: Early stopping at epoch 53
===== Split 6 =====
Train: 2031 → 2037
Val  : 2037 → 2040
Train size: 1825 | Val size: 1095
MAE: 1.787, RMSE: 2.282

Split 7: Early stopping at epoch 61
===== Split 7 =====
Train: 2037 → 2042
Val  : 

In [21]:
print("="*30)
print(f"Final Combined Results ({len(all_mae)} Splits):")
print(f"Average MAE  : {np.mean(all_mae):.4f} ± {np.std(all_mae):.4f}")
print(f"Average RMSE : {np.mean(all_rmse):.4f} ± {np.std(all_rmse):.4f}")
print("="*30)

Final Combined Results (8 Splits):
Average MAE  : 1.8913 ± 0.2466
Average RMSE : 2.3851 ± 0.2747


In [22]:
# predict
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, max_error

test_data_005 = read_data_005[(read_data_005['year'] > 2050) & (read_data_005['year'] <= 2080)].copy()
X_test_raw = test_data_005[features].values
y_test_raw = test_data_005['TREFMXAV_U'].values

X_test_scaled = scaler_X.transform(X_test_raw)

X_test_t, y_test_t = create_sequences(X_test_scaled, y_test_raw, window_size)

X_test_tensor = torch.FloatTensor(X_test_t)
y_test_tensor = torch.FloatTensor(y_test_t).view(-1, 1)


model.eval()

with torch.no_grad():
    y_pred_tensor = model(X_test_tensor)
    y_true = y_test_tensor.numpy().flatten()
    y_pred = y_pred_tensor.numpy().flatten()


r2 = r2_score(y_true, y_pred)
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
max_err = max_error(y_true, y_pred)

print("\n" + "="*35)
print(f"Test Evaluation (2051 - 2080):")
print(f"{'R-Squared':<15}: {r2:.4f}")
print(f"{'MAE':<15}: {mae:.4f}")
print(f"{'RMSE':<15}: {rmse:.4f}")
print(f"{'Max Error':<15}: {max_err:.4f}")
print("="*35)


Test Evaluation (2051 - 2080):
R-Squared      : 0.2804
MAE            : 3.7963
RMSE           : 4.5382
Max Error      : 14.3719


# LSTM
hidden size=32

window size=30

learning rate=0.001

epoch=200

patience=15

In [11]:
class TemperatureLSTM(nn.Module):
    def __init__(self, input_channels, hidden_size=32, num_layers=2):
        super(TemperatureLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(input_channels, hidden_size, num_layers, 
                            batch_first=True, dropout=0.2 if num_layers > 1 else 0)


        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = x.transpose(1, 2) 
        
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        out, _ = self.lstm(x, (h0, c0))
        
        out = out[:, -1, :]
        
        out = self.fc(out)
        return out

In [23]:
# Training
window_size = 30
input_channels = X_scaled_005.shape[1]
trained_models_lstm=[]

all_mae_lstm = []
all_rmse_lstm = []

for i, (X_tr, y_tr, X_va, y_va, t_s, t_e, v_s, v_e) in enumerate(splits_005):
    X_train_t, y_train_t = create_sequences(X_tr, y_tr, window_size)
    X_val_t, y_val_t = create_sequences(X_va, y_va, window_size)
    train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)
    
    model = TemperatureLSTM(input_channels, hidden_size=32, num_layers=2)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.L1Loss()
    early_stopping = EarlyStopping(patience=15, verbose=False)

    for epoch in range(200):
        model.train()
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
        
        model.eval()
        with torch.no_grad():
            val_out = model(X_val_t)
            val_mae = criterion(val_out, y_val_t).item()
        
        early_stopping(val_mae, model)
        if early_stopping.early_stop:
            break

    model.load_state_dict(early_stopping.best_state_dict)
    model.eval()
    trained_models_lstm.append(model)
    with torch.no_grad():
        preds = model(X_val_t)
        final_mae = criterion(preds, y_val_t).item()
        final_rmse = np.sqrt(F.mse_loss(preds, y_val_t).item())
        all_mae_lstm.append(final_mae)
        all_rmse_lstm.append(final_rmse)

    print(f"===== Split {i+1} =====")
    print(f"Train: {int(t_s)} → {int(t_e)}")
    print(f"Val  : {int(v_s)} → {int(v_e)}")
    print(f"Train size: {len(X_tr)} | Val size: {len(X_va)}")
    print(f"MAE: {mae:.3f}, RMSE: {rmse:.3f}")
    print()

===== Split 1 =====
Train: 2006 → 2011
Val  : 2011 → 2014
Train size: 1825 | Val size: 1095
MAE: 3.796, RMSE: 4.538

===== Split 2 =====
Train: 2011 → 2016
Val  : 2016 → 2019
Train size: 1825 | Val size: 1095
MAE: 3.796, RMSE: 4.538

===== Split 3 =====
Train: 2016 → 2021
Val  : 2021 → 2024
Train size: 1825 | Val size: 1095
MAE: 3.796, RMSE: 4.538

===== Split 4 =====
Train: 2021 → 2026
Val  : 2026 → 2029
Train size: 1825 | Val size: 1095
MAE: 3.796, RMSE: 4.538

===== Split 5 =====
Train: 2026 → 2031
Val  : 2031 → 2035
Train size: 1825 | Val size: 1095
MAE: 3.796, RMSE: 4.538

===== Split 6 =====
Train: 2031 → 2037
Val  : 2037 → 2040
Train size: 1825 | Val size: 1095
MAE: 3.796, RMSE: 4.538

===== Split 7 =====
Train: 2037 → 2042
Val  : 2042 → 2045
Train size: 1825 | Val size: 1095
MAE: 3.796, RMSE: 4.538

===== Split 8 =====
Train: 2042 → 2047
Val  : 2047 → 2050
Train size: 1825 | Val size: 1095
MAE: 3.796, RMSE: 4.538



In [24]:
print("="*30)
print(f"Final Combined Results ({len(all_mae_lstm)} Splits):")
print(f"Average MAE  : {np.mean(all_mae_lstm):.4f} ± {np.std(all_mae_lstm):.4f}")
print(f"Average RMSE : {np.mean(all_rmse_lstm):.4f} ± {np.std(all_rmse_lstm):.4f}")
print("="*30)

Final Combined Results (8 Splits):
Average MAE  : 1.5113 ± 0.0559
Average RMSE : 1.9360 ± 0.0726


In [25]:
# predict
model.eval()

with torch.no_grad():
    test_preds_tensor = model(X_test_tensor)
    y_true = y_test_tensor.numpy().flatten()
    y_pred = test_preds_tensor.numpy().flatten()

r2_lstm = r2_score(y_true, y_pred)
mae_lstm = mean_absolute_error(y_true, y_pred)
rmse_lstm = np.sqrt(mean_squared_error(y_true, y_pred))
max_err_lstm = max_error(y_true, y_pred)

print("\n" + "="*35)
print(f"LSTM Final Evaluation (2051 - 2080):")
print(f"{'R-Squared':<15}: {r2_lstm:.4f}")
print(f"{'MAE':<15}: {mae_lstm:.4f}")
print(f"{'RMSE':<15}: {rmse_lstm:.4f}")
print(f"{'Max Error':<15}: {max_err_lstm:.4f}")
print("="*35)


LSTM Final Evaluation (2051 - 2080):
R-Squared      : 0.8555
MAE            : 1.6033
RMSE           : 2.0339
Max Error      : 8.9707


# Transformer
d_model=32

window size=30

learning rate=0.001

epoch=200

patience=15

In [26]:
import math

class TemperatureTransformer(nn.Module):
    def __init__(self, input_channels, window_size, d_model=32, nhead=4, num_layers=1, dropout=0.1):
        super(TemperatureTransformer, self).__init__()
        
        self.input_fc = nn.Linear(input_channels, d_model)
        
        self.pos_encoder = nn.Parameter(torch.zeros(1, window_size, d_model))
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=d_model*2, 
            dropout=dropout, 
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.output_fc = nn.Sequential(
            nn.Linear(d_model * window_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        
        x = self.input_fc(x) + self.pos_encoder
        
        x = self.transformer_encoder(x)
        
        x = x.reshape(x.size(0), -1)
        
        x = self.output_fc(x)
        return x

In [27]:
# Training
window_size = 30 
input_channels = X_scaled_005.shape[1]
trained_models_trans=[]

all_mae_trans = []
all_rmse_trans = []

for i, (X_tr, y_tr, X_va, y_va, t_s, t_e, v_s, v_e) in enumerate(splits_005):
    X_train_t, y_train_t = create_sequences(X_tr, y_tr, window_size)
    X_val_t, y_val_t = create_sequences(X_va, y_va, window_size)
    train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)
    
    model = TemperatureTransformer(input_channels, window_size, d_model=32, nhead=4, num_layers=2)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.L1Loss()
    early_stopping = EarlyStopping(patience=15, verbose=False)

    for epoch in range(200):
        model.train()
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
        
        model.eval()
        with torch.no_grad():
            val_out = model(X_val_t)
            val_mae = criterion(val_out, y_val_t).item()
        
        early_stopping(val_mae, model)
        if early_stopping.early_stop:
            break

    model.load_state_dict(early_stopping.best_state_dict)
    model.eval()
    trained_models_trans.append(model)
    with torch.no_grad():
        preds = model(X_val_t)
        mae_val = criterion(preds, y_val_t).item()
        rmse_val = np.sqrt(F.mse_loss(preds, y_val_t).item())
        all_mae_trans.append(mae_val)
        all_rmse_trans.append(rmse_val)

    print(f"===== Split {i+1} =====")
    print(f"Train: {int(t_s)} → {int(t_e)}")
    print(f"Val  : {int(v_s)} → {int(v_e)}")
    print(f"Train size: {len(X_tr)} | Val size: {len(X_va)}")
    print(f"MAE: {mae:.3f}, RMSE: {rmse:.3f}")
    print()

===== Split 1 =====
Train: 2006 → 2011
Val  : 2011 → 2014
Train size: 1825 | Val size: 1095
MAE: 3.796, RMSE: 4.538

===== Split 2 =====
Train: 2011 → 2016
Val  : 2016 → 2019
Train size: 1825 | Val size: 1095
MAE: 3.796, RMSE: 4.538

===== Split 3 =====
Train: 2016 → 2021
Val  : 2021 → 2024
Train size: 1825 | Val size: 1095
MAE: 3.796, RMSE: 4.538

===== Split 4 =====
Train: 2021 → 2026
Val  : 2026 → 2029
Train size: 1825 | Val size: 1095
MAE: 3.796, RMSE: 4.538

===== Split 5 =====
Train: 2026 → 2031
Val  : 2031 → 2035
Train size: 1825 | Val size: 1095
MAE: 3.796, RMSE: 4.538

===== Split 6 =====
Train: 2031 → 2037
Val  : 2037 → 2040
Train size: 1825 | Val size: 1095
MAE: 3.796, RMSE: 4.538

===== Split 7 =====
Train: 2037 → 2042
Val  : 2042 → 2045
Train size: 1825 | Val size: 1095
MAE: 3.796, RMSE: 4.538

===== Split 8 =====
Train: 2042 → 2047
Val  : 2047 → 2050
Train size: 1825 | Val size: 1095
MAE: 3.796, RMSE: 4.538



In [28]:
print("="*30)
print(f"Final Combined Results ({len(all_mae_trans)} Splits):")
print(f"Average MAE  : {np.mean(all_mae_trans):.4f} ± {np.std(all_mae_trans):.4f}")
print(f"Average RMSE : {np.mean(all_rmse_trans):.4f} ± {np.std(all_rmse_trans):.4f}")
print("="*30)

Final Combined Results (8 Splits):
Average MAE  : 1.6935 ± 0.0872
Average RMSE : 2.1532 ± 0.0957


In [30]:
# predict
model.eval()

with torch.no_grad():
    y_pred_tensor = model(X_test_tensor)
    y_true = y_test_tensor.numpy().flatten()
    y_pred = y_pred_tensor.numpy().flatten()

r2_trans = r2_score(y_true, y_pred)
mae_trans = mean_absolute_error(y_true, y_pred)
rmse_trans = np.sqrt(mean_squared_error(y_true, y_pred))
max_err_trans = max_error(y_true, y_pred)

print("="*35)
print(" Transformer Performance (2051-2080) ")
print(f"{'R-Squared':<15}: {r2_trans:.4f}")
print(f"{'MAE':<15}: {mae_trans:.4f}")
print(f"{'RMSE':<15}: {rmse_trans:.4f}")
print(f"{'Max Error':<15}: {max_err_trans:.4f}")
print("="*35)

 Transformer Performance (2051-2080) 
R-Squared      : 0.7843
MAE            : 1.9764
RMSE           : 2.4846
Max Error      : 11.7967
